# 📊 Notebook 2 — Model Comparison

**Author:** Maulik Mathur | **HuggingFace:** [maulik78](https://huggingface.co/maulik78)

---

## The Question This Notebook Answers

> "How much does text understanding actually help with price prediction?"

We answer this by training 6 models of increasing sophistication
and measuring exactly how much each step improves accuracy.

## Why Start With Simple Models?

Before building a complex LLM pipeline, we need to know:
- What's the absolute baseline? (random guessing)
- Does the average price beat random? (almost always yes)
- Do simple features like weight help? (a little)
- Does the actual product text help? (significantly)
- How much does model complexity matter given the same text features?

**If XGBoost couldn't beat constant prediction, training an LLM would be pointless.**
Baselines tell us whether the problem is worth solving with complexity.

## The Progression
```
Model 1: Random          → $287 MAE  (knows nothing)
Model 2: Mean            → $141 MAE  (knows the distribution)
Model 3: Linear (3 feat) → $120 MAE  (uses weight + text length)
Model 4: Linear (BoW)    → $100 MAE  (reads the words)
Model 5: Random Forest   →  $85 MAE  (non-linear word patterns)
Model 6: XGBoost         →  $68 MAE  (gradient boosting, full data)
```

**Fine-tuned LLM (Notebook 3) achieves $58.74 MAE — 13.9% better than XGBoost**
because it understands what words MEAN, not just which words appear.

## 2.1 — Setup

In [ ]:
!pip install -q xgboost scikit-learn datasets huggingface_hub

import os
import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dataclasses import dataclass
from typing import Optional
from datasets import load_dataset
from huggingface_hub import login
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from google.colab import userdata
login(userdata.get('HF_TOKEN'), add_to_git_credential=True)

print('✅ Setup complete')

## 2.2 — Load the Preprocessed Dataset

We use the **preprocessed** dataset where the `summary` field
contains LLM-cleaned product descriptions.

**Why use summaries instead of raw text?**
The raw `text` field is noisy — HTML artifacts, part numbers,
duplicate marketing copy. The LLM-generated `summary` is clean
and structured, which helps all models downstream.

In [ ]:
# Using full dataset for real results
# Change to items_lite for fast testing
DATASET = 'maulik78/items_full'

ds    = load_dataset(DATASET)
train = list(ds['train'])
val   = list(ds['val'])
test  = list(ds['test'])

print(f'Dataset: {DATASET}')
print(f'  Train: {len(train):,}')
print(f'  Val:   {len(val):,}')
print(f'  Test:  {len(test):,}')
print(f'\nFields: {list(train[0].keys())}')
print(f'\nSample summary:')
print(train[0]['summary'])

## 2.3 — The Evaluation Function

Every model is evaluated using the **same function on the same test items**.
This is the only way comparisons are fair.

**Metric: MAE (Mean Absolute Error)**
- Directly interpretable: "$68 average error"
- Not dominated by outliers (unlike MSE/RMSE)
- Matches business intuition: every dollar of error is equal

In [ ]:
EVAL_SIZE   = 200   # test on 200 items per model
EVAL_SEED   = 42    # same seed = same 200 items for every model
all_results = {}    # model_name → MAE

# Pre-select the 200 test items (same for all models)
random.seed(EVAL_SEED)
eval_indices = random.sample(range(len(test)), EVAL_SIZE)
eval_items   = [test[i] for i in eval_indices]

print(f'Evaluation set: {len(eval_items)} items')
print(f'Price range in eval set: ${min(p["price"] for p in eval_items):.2f} – ${max(p["price"] for p in eval_items):.2f}')


def evaluate(model_name: str, predict_fn) -> float:
    """
    Evaluate a pricing model on the pre-selected eval items.
    
    predict_fn: function(product_dict) → price (float or string)
    
    Handles both float and string outputs so we can evaluate
    traditional ML models AND LLM-based models the same way.
    """
    errors = []
    
    for item in eval_items:
        actual = item['price']
        
        try:
            output = predict_fn(item)
            
            # Parse output — handle float, int, or string like '$49.99'
            if isinstance(output, (int, float)):
                predicted = max(0.0, float(output))
            else:
                cleaned   = str(output).replace('$','').replace(',','').strip()
                match     = re.search(r'\d+\.?\d*', cleaned)
                predicted = float(match.group()) if match else 0.0
        except:
            predicted = 0.0
        
        errors.append(abs(predicted - actual))
    
    mae = sum(errors) / len(errors)
    all_results[model_name] = mae
    print(f'  {model_name:<40} MAE: ${mae:.2f}')
    return mae


print('\nEvaluation function ready.')

## 2.4 — Model 1: Random Pricer

**What it does:** Returns a random price between $1 and $999.
Completely ignores the product.

**Purpose:** The absolute floor. If any model can't beat this, it's broken.

**Expected MAE:** ~$285 (roughly 1/3 of the full $0-$999 range)

In [ ]:
def random_pricer(item):
    """Predict a random price $1-$999. Ignores the item entirely."""
    return random.randrange(1, 1000)

random.seed(EVAL_SEED)
evaluate('Random Pricer', random_pricer)

## 2.5 — Model 2: Constant Pricer

**What it does:** Always predicts the training set mean price.
Ignores the specific product.

**Purpose:** Tests if knowing the price distribution helps.
Represents a model that learned WHAT prices exist but not WHY.

**Key question this answers:** Is there enough variance in prices
that we need to look at individual products? (Yes — see the MAE improvement)

In [ ]:
# Compute mean price from training data
train_prices = [item['price'] for item in train]
mean_price   = sum(train_prices) / len(train_prices)

print(f'Training set price statistics:')
print(f'  Mean:   ${mean_price:.2f}')
print(f'  Median: ${sorted(train_prices)[len(train_prices)//2]:.2f}')
print(f'  Std:    ${np.std(train_prices):.2f}')
print()

def constant_pricer(item):
    """Always return the training mean. Ignores individual products."""
    return mean_price

evaluate('Constant Pricer (mean)', constant_pricer)

## 2.6 — Model 3: Linear Regression with 3 Features

**What it does:** Uses 3 numerical features — weight, whether weight
is missing, and text length.

**Purpose:** Tests if simple structural features contain price signal.

**What the coefficients mean:**
- A positive weight coefficient confirms heavier = more expensive
- A positive text_length coefficient suggests detailed = more expensive
- The model learns the best linear combination of these 3 numbers

In [ ]:
def get_numerical_features(item: dict) -> dict:
    """Extract 3 numerical features from a product."""
    return {
        'weight':         item.get('weight') or 0.0,
        'weight_missing': 1 if (item.get('weight') or 0) == 0 else 0,
        'text_length':    len(item.get('summary') or item.get('text', '')),
    }

# Build train/test feature matrices
X_train_num = pd.DataFrame([get_numerical_features(p) for p in train])
y_train     = [p['price'] for p in train]

# Train
lr_num = LinearRegression()
lr_num.fit(X_train_num, y_train)

# Show what the model learned
print('Learned relationship (coefficient = price change per unit):')
for col, coef in zip(X_train_num.columns, lr_num.coef_):
    direction = '↑ heavier = more expensive' if col == 'weight' and coef > 0 else ''
    print(f'  {col:<20} ${coef:>8.3f}  {direction}')
print(f'  {"base price":<20} ${lr_num.intercept_:>8.3f}')

def linear_numerical_pricer(item):
    """Predict price using 3 numerical features."""
    X = pd.DataFrame([get_numerical_features(item)])
    return max(0, lr_num.predict(X)[0])

print()
evaluate('Linear Regression (3 features)', linear_numerical_pricer)

## 2.7 — Bag of Words Vectorization

For the next 3 models, we convert product text to numerical vectors.

**CountVectorizer** builds a vocabulary of 2,000 most common words
and represents each product as: "how many times does each word appear?"

```
Text: "Sony noise cancelling wireless headphones 30 hour battery"
Vector: [0, 0, 1, 0, 1, 0, 0, 1, 1, 0, ...]
                 ↑      ↑      ↑  ↑
              'sony' 'noise' 'wireless' 'battery'
```

**Limitation:** Treats words independently. Doesn't understand
that "Sony + noise-cancelling" together means premium product.

In [ ]:
# Build vocabulary from training data
# max_features=2000: top 2000 most common words
# stop_words='english': ignore 'the', 'a', 'is' etc.
train_docs = [p.get('summary') or p.get('text','') for p in train]
train_y    = np.array([p['price'] for p in train])

vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X_train_bow = vectorizer.fit_transform(train_docs)

vocab = vectorizer.get_feature_names_out()
print(f'Vocabulary: {len(vocab):,} words')
print(f'Feature matrix: {X_train_bow.shape[0]:,} items × {X_train_bow.shape[1]:,} word features')
print(f'Matrix is {100*(1 - X_train_bow.nnz/(X_train_bow.shape[0]*X_train_bow.shape[1])):.1f}% zeros')
print(f'\nSample vocabulary words: {vocab[400:410].tolist()}')

## 2.8 — Model 4: Linear Regression on Bag of Words

**What it does:** Uses all 2,000 word count features.
The model learns: which words correlate with higher/lower price?

**Big improvement over Model 3** because now we actually
read the product description.

In [ ]:
# Train on word count features
lr_bow = LinearRegression()
lr_bow.fit(X_train_bow, train_y)

# Show which words most influence price predictions
coefs        = lr_bow.coef_
top_positive = np.argsort(coefs)[-8:][::-1]
top_negative = np.argsort(coefs)[:8]

print('Words that INCREASE predicted price:')
for idx in top_positive:
    print(f'  "{vocab[idx]}" → +${coefs[idx]:.0f}')

print('\nWords that DECREASE predicted price:')
for idx in top_negative:
    print(f'  "{vocab[idx]}" → -${abs(coefs[idx]):.0f}')

def bow_linear_pricer(item):
    """Predict price using 2000 word count features with Linear Regression."""
    text = item.get('summary') or item.get('text', '')
    x    = vectorizer.transform([text])
    return max(0, lr_bow.predict(x)[0])

print()
evaluate('NL Linear Regression (BoW)', bow_linear_pricer)

## 2.9 — Model 5: Random Forest

**What it does:** 100 decision trees, each trained on a random
subset of data AND features. Final prediction = average of all trees.

**Why better than Linear Regression on BoW:**
Captures interactions between words. Linear regression treats
each word independently. Random Forest can learn:
"if 'sony' AND 'headphones' appear → predict premium price"

**Note:** We only use 15k items because full RF takes 15+ hours.

In [ ]:
RF_SUBSET = 15_000
print(f'Training Random Forest on {RF_SUBSET:,} items (subset due to compute time)...')

rf = RandomForestRegressor(
    n_estimators=100,  # 100 trees
    random_state=42,
    n_jobs=-1          # use all CPU cores
)
rf.fit(X_train_bow[:RF_SUBSET], train_y[:RF_SUBSET])
print('✅ Random Forest trained')

def random_forest_pricer(item):
    """Predict price using ensemble of 100 decision trees."""
    text = item.get('summary') or item.get('text', '')
    x    = vectorizer.transform([text])
    return max(0, rf.predict(x)[0])

evaluate('Random Forest (15k subset)', random_forest_pricer)

## 2.10 — Model 6: XGBoost (Our Best Traditional ML Model)

**What it does:** 1000 trees built sequentially.
Each tree corrects errors of all previous trees.

**The gradient boosting idea:**
```
Start: predict mean price ($141) for everything
Tree 1: learn to fix those errors
Tree 2: learn to fix Tree 1's errors
...
Tree 1000: fix Tree 999's errors
Final: sum of all small corrections
```

**Trains on the FULL 800k dataset** (unlike RF which used 15k).
This is our traditional ML benchmark: **$68.23 MAE**

In [ ]:
print(f'Training XGBoost on {len(train):,} items...')
print('(~20-30 minutes on CPU)')

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,   # 1000 sequential trees
    learning_rate=0.1,   # each tree corrects 10% of remaining error
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_bow, train_y)
print('✅ XGBoost trained')

def xgboost_pricer(item):
    """Predict price using XGBoost gradient boosting."""
    text = item.get('summary') or item.get('text', '')
    x    = vectorizer.transform([text])
    return max(0, xgb_model.predict(x)[0])

evaluate('XGBoost (gradient boosting)', xgboost_pricer)

## 2.11 — Feature Importance: What Words Drive Price?

XGBoost tracks how much each feature reduced prediction error.
This reveals WHICH words contain the most price signal —
a unique insight not available from neural approaches.

In [ ]:
importances = xgb_model.feature_importances_
top_20      = np.argsort(importances)[-20:][::-1]

plt.figure(figsize=(14, 6))
plt.title('Top 20 Words That Predict Price (XGBoost Feature Importance)', fontsize=13)
colors = ['#2196F3' if importances[i] > importances[top_20[5]] else '#90CAF9' for i in top_20]
plt.bar(range(20), importances[top_20], color=colors, edgecolor='white')
plt.xticks(range(20), [vocab[i] for i in top_20], rotation=45, ha='right', fontsize=11)
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 price-predictive words:')
for rank, idx in enumerate(top_20[:10], 1):
    print(f'  {rank:>2}. "{vocab[idx]}" — importance: {importances[idx]:.4f}')

## 2.12 — Why XGBoost Still Has Limits (The Case for LLMs)

This cell demonstrates the fundamental limitation of BoW approaches.

In [ ]:
# Demonstrate XGBoost's limitation with a concrete example
test_descriptions = [
    {
        'summary': 'Title: Sony WH-1000XM4 Headphones\nCategory: Electronics\nBrand: Sony\nDescription: Industry-leading noise cancelling wireless headphones\nDetails: 30 hour battery, touch controls, multipoint connection',
        'expected': ~278,
        'label': 'Premium Sony headphones'
    },
    {
        'summary': 'Title: Generic Bluetooth Headphones\nCategory: Electronics\nBrand: Unknown\nDescription: Wireless headphones with noise cancelling\nDetails: 20 hour battery, basic controls',
        'expected': ~25,
        'label': 'Generic headphones (same words, much cheaper)'
    },
]

print('XGBoost limitation — same words, very different prices:\n')
for desc in test_descriptions:
    pred = xgboost_pricer(desc)
    print(f'  {desc["label"]}')
    print(f'    Expected: ~${desc["expected"]}')
    print(f'    XGBoost:   ${pred:.2f}')
    print()

print('Both descriptions contain: headphones, wireless, noise, cancelling, battery')
print('XGBoost sees similar word counts → predicts similar prices')
print('An LLM understands BRAND CONTEXT: Sony = premium, Unknown = budget')

## 2.13 — Complete Results

In [ ]:
# Plot all results
sorted_results = sorted(all_results.items(), key=lambda x: x[1], reverse=True)
names  = [r[0] for r in sorted_results]
maes   = [r[1] for r in sorted_results]
colors = ['#F44336' if m > 200 else '#FF9800' if m > 100 else
          '#FFC107' if m > 75 else '#4CAF50' for m in maes]

plt.figure(figsize=(13, 6))
plt.title('Model Comparison — Mean Absolute Error (lower is better)', fontsize=13)
bars = plt.bar(range(len(names)), maes, color=colors, edgecolor='white', width=0.6)
for bar, mae in zip(bars, maes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'${mae:.0f}', ha='center', fontsize=10, fontweight='bold')
plt.xticks(range(len(names)), names, rotation=30, ha='right', fontsize=10)
plt.ylabel('MAE ($)')
plt.axhline(y=maes[-1], color='green', linestyle='--', alpha=0.5, label='XGBoost baseline')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print table
random_mae = all_results.get('Random Pricer', 287)
print('\n' + '='*65)
print('  NOTEBOOK 2 COMPLETE — MODEL COMPARISON')
print('='*65)
print(f'  {"Model":<40} {"MAE ($)":>8}  {"vs Random":>10}')
print('-'*65)
for name, mae in sorted_results:
    improvement = random_mae - mae
    print(f'  {name:<40} ${mae:>7.2f}  -{improvement:>7.0f}')
print('='*65)
print(f'\n  Best traditional ML: XGBoost at ${all_results["XGBoost (gradient boosting)"]:.2f} MAE')
print(f'  Fine-tuned LLM (Notebook 3): ~$58.74 MAE (13.9% better)')
print(f'\n  WHY LLM IS BETTER:')
print(f'  XGBoost sees word COUNTS independently')
print(f'  LLM understands MEANING — brand, quality tier, use case')
print(f'\n➡️  Next: Notebook 3 — Fine-tuning Llama 3.2-3B with QLoRA')